# Faza 3 — minimalni srpski Dataset adapter


## Goal

Proveri srpski alfabet i tab-separated anotacije, eksplicitni speaker-disjoint
split, runtime discovery i batch sa dva različito duga klipa. Izlaz ostaje VIPL
rečnik: `vid`, `txt`, `txt_len`, `vid_len`.


## Setup

Ova faza je lagana i radi na CPU-u. Potrebni su Phase 2 ZIP i originalni `processed.zip`.


In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'editdistance>=0.8.1', 'opencv-python-headless>=4.10', 'pytest>=8.4'],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'editdistance>=0.8.1', 'opencv-python-headless>=4.10', 'pytest>=8.4'], returncode=0)

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/nikolabakic/Vizuelno-prepoznavanje-govora-na-osnovu-pokreta-usana-pomo-u-LipNet-modela.git'
REPO = Path('/content/lipnet-serbian')
if not (REPO / 'lipnet').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
print('Repo:', REPO)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


Repo: /content/lipnet-serbian
Commit: 7f2e16d867b87fb75f779dc12504161360454539


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/LipNet')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive izlaz:', DRIVE_ROOT)


Mounted at /content/drive
Drive izlaz: /content/drive/MyDrive/LipNet


### 0. Pokreni lake kompatibilnosne testove (bez model forward-a)


In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', 'tests/test_vipl_phases.py'],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q', 'tests/test_vipl_phases.py'], returncode=0)

## Steps

### 1. Raspakuj mouth frejmove i samo ALIGN anotacije na lokalni disk


In [ ]:
import zipfile

MOUTH_ARCHIVE = DRIVE_ROOT / 'ai_speak_lip.zip'
SOURCE_ARCHIVE = Path('/content/drive/MyDrive/processed.zip')  # promeni po potrebi
MOUTH_ROOT = Path('/content/ai_speak_lip')
ALIGN_EXTRACT = Path('/content/ai_speak_align')
assert MOUTH_ARCHIVE.exists() and SOURCE_ARCHIVE.exists()

if not next(MOUTH_ROOT.glob('spk*/video/video_a/*'), None):
    MOUTH_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(MOUTH_ARCHIVE) as archive:
        archive.extractall(MOUTH_ROOT)
if not next(ALIGN_EXTRACT.rglob('spk*/alignment/*.align'), None):
    ALIGN_EXTRACT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
        members = [
            member for member in archive.infolist()
            if '/alignment/' in f'/{member.filename}' and member.filename.endswith('.align')
        ]
        archive.extractall(ALIGN_EXTRACT, members=members)
first_alignment = next(ALIGN_EXTRACT.rglob('spk*/alignment/*.align'))
CORPUS_ROOT = first_alignment.parents[2]
print('Mouth root:', MOUTH_ROOT)
print('Annotation root:', CORPUS_ROOT)


Mouth root: /content/ai_speak_lip
Annotation root: /content/ai_speak_align/processed


### 2. Proveri split, alfabet, parser i CTC round-trip


In [ ]:
from data.splits import SPLITS, TRAIN_SPEAKERS
from lipnet.dataset import (
    SERBIAN_LETTERS, SerbianDataset, parse_ai_speak_alignment,
)

split_sets = [set(SPLITS[name]) for name in ('train', 'validation', 'test')]
assert not (split_sets[0] & split_sets[1] | split_sets[0] & split_sets[2] | split_sets[1] & split_sets[2])
assert 1 + len(SERBIAN_LETTERS) == 29  # blank + 28 vidljivih simbola

alphabet_text = ''.join(SERBIAN_LETTERS).strip()
encoded = SerbianDataset.txt2arr(alphabet_text, start=1)
assert SerbianDataset.arr2txt(encoded, start=1) == alphabet_text

example_align = next(CORPUS_ROOT.glob(f'{TRAIN_SPEAKERS[0]}/alignment/*.align'))
example_text = parse_ai_speak_alignment(example_align)
assert 'sil' not in example_text.split() and 'sp' not in example_text.split()
print('Klase:', 1 + len(SERBIAN_LETTERS))
print('Primer:', example_align.name, '->', example_text)
print('Split:', {name: len(speakers) for name, speakers in SPLITS.items()})


Klase: 29
Primer: spk01_028.align -> kraj m napred j nedelja sedam
Split: {'train': 16, 'validation': 3, 'test': 3}


### 3. Napravi Dataset i batch sa dve različite dužine


In [ ]:
from torch.utils.data import DataLoader
from lipnet.dataset import variable_length_collate, validate_ctc_batch

train_dataset = SerbianDataset(
    video_path=MOUTH_ROOT,
    anno_path=CORPUS_ROOT,
    speakers=TRAIN_SPEAKERS,
    phase='train',
)
first = train_dataset[0]
second = None
for index in range(1, len(train_dataset)):
    candidate = train_dataset[index]
    if candidate['vid_len'] != first['vid_len']:
        second = candidate
        break
assert second is not None, 'Svi klipovi neočekivano imaju istu dužinu.'
batch = variable_length_collate([first, second])
assert set(batch) == {'vid', 'txt', 'txt_len', 'vid_len'}
assert batch['vid_len'][0] != batch['vid_len'][1]
assert batch['vid'].shape[2] == int(batch['vid_len'].max())
validate_ctc_batch(batch, batch['vid_len'])
print('Broj train primera:', len(train_dataset))
print({key: tuple(value.shape) for key, value in batch.items()})
print('vid_len:', batch['vid_len'].tolist(), 'txt_len:', batch['txt_len'].tolist())

loader = DataLoader(
    train_dataset, batch_size=2, shuffle=False, num_workers=0,
    collate_fn=variable_length_collate,
)
assert set(next(iter(loader))) == {'vid', 'txt', 'txt_len', 'vid_len'}


Broj train primera: 2877
{'vid': (2, 3, 132, 64, 128), 'txt': (2, 30), 'txt_len': (2,), 'vid_len': (2,)}
vid_len: [130, 132] txt_len: [27, 30]


## Checks

### 4. Proveri normalizaciju, padding i sačuvaj audit


In [ ]:
import json
assert 0.0 <= float(batch['vid'].min()) <= float(batch['vid'].max()) <= 1.0
for index, length in enumerate(batch['vid_len']):
    tail = batch['vid'][index, :, int(length):]
    assert tail.numel() == 0 or float(tail.abs().sum()) == 0.0

result = {
    'phase': 3,
    'num_classes': 1 + len(SERBIAN_LETTERS),
    'train_samples': len(train_dataset),
    'batch_shape': list(batch['vid'].shape),
    'vid_len': batch['vid_len'].tolist(),
    'txt_len': batch['txt_len'].tolist(),
    'keys': sorted(batch),
}
result_path = DRIVE_ROOT / 'phase3_dataset_audit.json'
result_path.write_text(json.dumps(result, indent=2) + '\n', encoding='utf-8')
print(json.dumps(result, indent=2))


{
  "phase": 3,
  "num_classes": 29,
  "train_samples": 2877,
  "batch_shape": [
    2,
    3,
    132,
    64,
    128
  ],
  "vid_len": [
    130,
    132
  ],
  "txt_len": [
    27,
    30
  ],
  "keys": [
    "txt",
    "txt_len",
    "vid",
    "vid_len"
  ]
}


## Next Steps

Faza 3 prolazi kada su govornici disjunktni, round-trip čuva sva slova, dva
različita klipa dele padded batch i CTC feasibility provera ne prijavi grešku.
Faza 4 koristi isti batch ugovor na GPU-u.
